### Four-model held-out validation on the NJ2 dataset

- data: NJ2 quality ratings

- validation design: one blocked held-out-date cross-validation, with approximately 20% of calendar dates held out and all raters on those dates withheld together

- comparison design: four manuscript model variants formed by crossing two latent structures with two rating layers

- latent structures:

  - Spatial: $\theta_n = a_{j[n]} + u_{s[n]}$

  - Seasonality: $\theta_n = a_{j[n]} + h_{j[n]}(t_{e[n]}) + u_{s[n]}$

- rating layers:

  - original rating: shared thresholds `tau` plus event-specific adjustment `beta[event]`

  - rater thresholds: rater-specific thresholds `tau_r`

- models compared: Spatial + original rating, Spatial + rater thresholds, Seasonality + original rating, Seasonality + rater thresholds

- shared comparison metrics: mean and total held-out log predictive score, exact accuracy, and adjacent accuracy

- manuscript diagnostic figure: observed versus posterior predictive held-out rating frequencies for categories 1-9 from the full Seasonality + rater-threshold model

- caveat: Rater A appears only on held-out dates, so rows from Rater A are excluded from the rater-threshold posterior predictive diagnostic.

In [25]:
# general-use imports
from pathlib import Path
import numpy as np
import pickle
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
import arviz as az
from scipy.special import logsumexp
from scipy.stats import mode

# custom imports
from settings import DATA_DIR, ROOT_DIR
from nteprsm import utils
import rutils
rutils.set_custom_template()

In [21]:
TEST_PATH = Path(ROOT_DIR / "data/model_output/cross_val_nj2_quality_conservative/test_data.pkl")
TARGET_COL = "value"
MODEL_SPECS = {
    "Spatial + original rating": Path(ROOT_DIR / "data/model_output/cross_val_nj2_compare_spatial_old/fit_spatial_old_cross_val.pkl"),
    "Spatial + rater thresholds": Path(ROOT_DIR / "data/model_output/cross_val_nj2_compare_spatial_rater/fit_spatial_rater_cross_val.pkl"),
    "Seasonality + original rating": Path(ROOT_DIR / "data/model_output/cross_val_nj2_compare_annual_old/fit_annual_old_cross_val.pkl"),
    "Seasonality + rater thresholds": Path(ROOT_DIR / "data/model_output/cross_val_nj2_quality_conservative/fit_seasonality_nj2_quality_cross_val.pkl"),
}

with TEST_PATH.open("rb") as file:
    test_data = pickle.load(file).reset_index(drop=True)

y_true = test_data[TARGET_COL].to_numpy().astype(int)
category_labels = list(range(int(y_true.min()), int(y_true.max()) + 1))
display_category_labels = [category + 1 for category in category_labels]


def load_fit(fit_path):
    with fit_path.open("rb") as file:
        return pickle.load(file)


def summarize_model(model_name, fit_path):
    fit_data = load_fit(fit_path)
    y_rep = np.asarray(fit_data.stan_variable("y_rep_test"))
    y_pred = mode(y_rep, axis=0, keepdims=False).mode.astype(int)
    ll_test = np.asarray(fit_data.stan_variable("log_lik_test"))
    held_out_log_score = logsumexp(ll_test, axis=0) - np.log(ll_test.shape[0])
    summary = {
        "model": model_name,
        "mean_held_out_log_score": held_out_log_score.mean(),
        "total_held_out_log_score": held_out_log_score.sum(),
        "exact_accuracy": np.mean(y_pred == y_true),
        "adjacent_accuracy": np.mean(np.abs(y_pred - y_true) <= 1),
    }
    return summary


model_rows = [
    summarize_model(model_name, fit_path)
    for model_name, fit_path in MODEL_SPECS.items()
]
results_df = (
    pd.DataFrame(model_rows)
    .sort_values("mean_held_out_log_score", ascending=False)
    .reset_index(drop=True)
)

In [16]:
comparison_df = results_df.copy()
comparison_df[['mean_held_out_log_score', 'total_held_out_log_score', 'exact_accuracy', 'adjacent_accuracy']] = (
    comparison_df[['mean_held_out_log_score', 'total_held_out_log_score', 'exact_accuracy', 'adjacent_accuracy']]
    .round(3)
)
comparison_df

,model,mean_held_out_log_score,total_held_out_log_score,exact_accuracy,adjacent_accuracy
0,Seasonality + rater thresholds,-1.707,-3187.256,0.291,0.689
1,Spatial + rater thresholds,-1.757,-3280.701,0.281,0.664
2,Seasonality + original rating,-1.971,-3679.629,0.148,0.428
3,Spatial + original rating,-2.111,-3941.752,0.082,0.243


### Main-text posterior predictive category-frequency figure

This section creates a manuscript-ready posterior predictive check for the full Seasonality + rater-threshold model. It compares observed held-out rating frequencies with posterior predictive frequencies for rating categories 1-9. Rater A is excluded because that anonymized rater appears only in the held-out dates, leaving its rater-specific thresholds prior-only.

In [22]:
full_model_fit_path = MODEL_SPECS["Seasonality + rater thresholds"]

with full_model_fit_path.open("rb") as file:
    full_model_fit = pickle.load(file)

rater_a_mask = test_data["rater_code"].eq(1).to_numpy()
evaluation_mask = ~rater_a_mask

y_observed = test_data.loc[evaluation_mask, TARGET_COL].to_numpy().astype(int)
y_rep = np.asarray(full_model_fit.stan_variable("y_rep_test")).astype(int)[:, evaluation_mask]

rating_categories = np.arange(1, 10)
observed_counts = np.bincount(y_observed, minlength=9)[:9]
observed_freq = observed_counts / observed_counts.sum()

rep_counts = np.stack([
    np.bincount(draw, minlength=9)[:9]
    for draw in y_rep
])
rep_freq = rep_counts / y_rep.shape[1]
ppc_mean = rep_freq.mean(axis=0)
ppc_lower = np.quantile(rep_freq, 0.05, axis=0)
ppc_upper = np.quantile(rep_freq, 0.95, axis=0)

posterior_predictive_category_freq_df = pd.DataFrame({
    "category": rating_categories,
    "observed_count": observed_counts,
    "observed_freq": observed_freq,
    "ppc_mean_freq": ppc_mean,
    "ppc_q05_freq": ppc_lower,
    "ppc_q95_freq": ppc_upper,
    "observed_within_90pct_interval": (ppc_lower <= observed_freq) & (observed_freq <= ppc_upper),
})
posterior_predictive_category_freq_df.round(3)

,category,observed_count,observed_freq,ppc_mean_freq,ppc_q05_freq,ppc_q95_freq,observed_within_90pct_interval
0,1,66,0.041,0.029,0.022,0.038,False
1,2,101,0.063,0.047,0.038,0.057,False
2,3,170,0.106,0.120,0.106,0.135,True
3,4,327,0.204,0.193,0.176,0.211,True
4,5,319,0.199,0.210,0.191,0.229,True
5,6,290,0.181,0.191,0.172,0.209,True
6,7,205,0.128,0.135,0.120,0.151,True
7,8,97,0.061,0.063,0.052,0.074,True
8,9,25,0.016,0.011,0.006,0.016,True


In [31]:
reports_dir = Path(ROOT_DIR / "reports/manuscript2")
figure_path = reports_dir / "figure9_heldout_ppc_category_frequencies.png"

plot_df = posterior_predictive_category_freq_df.copy()
plot_df["category_label"] = plot_df["category"].astype(str)
plot_df["ppc_error_plus"] = plot_df["ppc_q95_freq"] - plot_df["ppc_mean_freq"]
plot_df["ppc_error_minus"] = plot_df["ppc_mean_freq"] - plot_df["ppc_q05_freq"]

observed_color = "blue"
posterior_predictive_color = "orange"
outline_color = "black"

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=plot_df["category_label"],
        y=plot_df["observed_freq"],
        name="Observed held-out frequency",
        marker=dict(
            color=observed_color,
            line=dict(color=outline_color, width=0.7),
        ),
        offsetgroup="observed",
    )
)
fig.add_trace(
    go.Bar(
        x=plot_df["category_label"],
        y=plot_df["ppc_mean_freq"],
        name="Posterior predictive mean (90% interval)",
        marker=dict(
            color=posterior_predictive_color,
            line=dict(color=outline_color, width=0.7),
        ),
        error_y=dict(
            type="data",
            array=plot_df["ppc_error_plus"],
            arrayminus=plot_df["ppc_error_minus"],
            visible=True,
            color=outline_color,
            thickness=1.0,
            width=3,
        ),
        offsetgroup="posterior_predictive",
    )
)

fig.update_layout(
    template="custom",
    width=680,
    height=460,
    bargap=0.18,
    bargroupgap=0.04,
    xaxis_title="Rating category",
    yaxis_title="Frequency",
    yaxis_range=[0, max(ppc_upper.max(), observed_freq.max()) * 1.18],
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.22,
        xanchor="center",
        x=0.5,
        traceorder="normal",
    ),
    margin=dict(l=70, r=25, t=20, b=110),
)

reports_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(figure_path, scale=3)
fig.show()

print(f"Saved PNG: {figure_path}")

Saved PNG: /Users/henryqu/Documents/GitHub/nteprsm/reports/manuscript2/figure9_heldout_ppc_category_frequencies.png
